<a href="https://colab.research.google.com/github/nahmeddn-sys/GENAI/blob/main/Assigmnet_4_Implement_QLoRA_find_tunning_of_a_model_from_scratch_preferably_a_commercial_LLM_like_mistral_or_Claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this assignment, QLoRA (Quantized Low-Rank Adaptation) fine-tuning was implemented on the large language model Mistral 7B using the free GPU available in Google Colab. Large Language Models typically require very high computational resources to train or fine-tune because they contain billions of parameters. QLoRA addresses this challenge by combining 4-bit model quantization with Low-Rank Adaptation (LoRA) so that only a small number of additional parameters are trained while the original model weights remain frozen. In this implementation, the pretrained Mistral model was loaded using 4-bit quantization with BitsAndBytes, and LoRA adapters were applied to selected attention layers using PEFT. A custom instruction dataset containing around 100 samples was prepared and formatted into an instruction-response structure, which allowed the model to learn task-specific responses during training. The training pipeline was executed using tools from Hugging Face Transformers and TRL, enabling efficient supervised fine-tuning. After training, the learned LoRA adapters were saved and used together with the base model for inference. This approach demonstrates how modern parameter-efficient techniques allow developers to adapt powerful language models on limited hardware while still achieving meaningful task-specific behavior.

**Install Libraries**

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets trl

**Import Libraries**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

**Load Dataset**

In [ ]:
data=[
{"instruction":"Explain what Artificial Intelligence is in simple words.","input":"","output":"Artificial Intelligence is the ability of machines to perform tasks that normally require human intelligence such as learning, reasoning, and problem solving."},
{"instruction":"What is Machine Learning?","input":"","output":"Machine Learning is a branch of artificial intelligence where computers learn patterns from data and improve their performance without being explicitly programmed."},
{"instruction":"Explain the difference between AI and Machine Learning.","input":"","output":"Artificial Intelligence is a broad field focused on building intelligent machines, while Machine Learning is a subset of AI that enables systems to learn from data."},
{"instruction":"What is Natural Language Processing?","input":"","output":"Natural Language Processing is a field of AI that allows computers to understand, interpret, and generate human language."},
{"instruction":"Explain what a dataset is.","input":"","output":"A dataset is a collection of structured or unstructured data used for analysis, training machine learning models, or research."},

{"instruction":"What is a programming language?","input":"","output":"A programming language is a formal language used to give instructions to a computer so it can perform specific tasks."},
{"instruction":"Explain Python in simple terms.","input":"","output":"Python is a popular programming language known for its simple syntax and wide use in web development, data science, and artificial intelligence."},
{"instruction":"What is an algorithm?","input":"","output":"An algorithm is a step by step set of instructions designed to solve a problem or perform a task."},
{"instruction":"Explain what debugging means.","input":"","output":"Debugging is the process of finding and fixing errors or bugs in a computer program."},
{"instruction":"What is open source software?","input":"","output":"Open source software is software whose source code is freely available for anyone to view, modify, and distribute."},

{"instruction":"What is cloud computing?","input":"","output":"Cloud computing allows users to access computing resources like servers and storage over the internet instead of using local machines."},
{"instruction":"Explain what a database is.","input":"","output":"A database is an organized collection of data that can be easily accessed, managed, and updated."},
{"instruction":"What is a server?","input":"","output":"A server is a computer or system that provides resources, data, or services to other computers called clients."},
{"instruction":"Explain the purpose of APIs.","input":"","output":"APIs allow different software systems to communicate and exchange data with each other."},
{"instruction":"What does GPU stand for?","input":"","output":"GPU stands for Graphics Processing Unit, a processor designed to handle graphics and parallel computations."},

{"instruction":"What is the internet?","input":"","output":"The internet is a global network of interconnected computers that communicate using standardized protocols."},
{"instruction":"Explain what a web browser is.","input":"","output":"A web browser is a software application used to access and view websites on the internet."},
{"instruction":"What is a URL?","input":"","output":"A URL is the address used to locate a specific resource or webpage on the internet."},
{"instruction":"Explain what HTML is.","input":"","output":"HTML is a markup language used to structure content on webpages."},
{"instruction":"What is CSS used for?","input":"","output":"CSS is used to style and design the layout of webpages."},

{"instruction":"Explain what cybersecurity means.","input":"","output":"Cybersecurity is the practice of protecting systems, networks, and data from digital attacks."},
{"instruction":"What is encryption?","input":"","output":"Encryption is the process of converting data into a secure format so that only authorized users can read it."},
{"instruction":"Explain what a password manager is.","input":"","output":"A password manager securely stores and manages passwords for different accounts."},
{"instruction":"What is phishing?","input":"","output":"Phishing is a cyberattack where attackers trick users into revealing sensitive information."},
{"instruction":"Explain two factor authentication.","input":"","output":"Two factor authentication adds an extra security step by requiring a second form of verification."},

{"instruction":"Explain what teamwork means.","input":"","output":"Teamwork is the process of working together with others to achieve a common goal."},
{"instruction":"Why is communication important in teams?","input":"","output":"Communication helps team members share ideas, coordinate tasks, and solve problems effectively."},
{"instruction":"Explain the concept of leadership.","input":"","output":"Leadership is the ability to guide, motivate, and influence people to achieve shared objectives."},
{"instruction":"What is time management?","input":"","output":"Time management is the process of planning and organizing how to spend time efficiently."},
{"instruction":"Explain why goal setting is important.","input":"","output":"Goal setting helps individuals focus their efforts and track progress toward desired outcomes."},

{"instruction":"What is climate change?","input":"","output":"Climate change refers to long term changes in global temperatures and weather patterns."},
{"instruction":"Explain renewable energy.","input":"","output":"Renewable energy comes from natural sources that are constantly replenished such as sunlight and wind."},
{"instruction":"What is solar energy?","input":"","output":"Solar energy is energy obtained from sunlight using solar panels."},
{"instruction":"Explain wind energy.","input":"","output":"Wind energy is generated by using wind turbines to convert wind into electricity."},
{"instruction":"Why is environmental protection important?","input":"","output":"Environmental protection helps preserve natural resources and ecosystems for future generations."},

{"instruction":"Explain what a startup is.","input":"","output":"A startup is a young company focused on developing innovative products or services."},
{"instruction":"What is innovation?","input":"","output":"Innovation is the process of creating new ideas, products, or methods that improve existing solutions."},
{"instruction":"Explain entrepreneurship.","input":"","output":"Entrepreneurship is the process of starting and managing a business venture."},
{"instruction":"What is productivity?","input":"","output":"Productivity refers to how efficiently tasks are completed using available resources."},
{"instruction":"Explain the concept of problem solving.","input":"","output":"Problem solving involves identifying an issue and finding effective solutions."},

{"instruction":"Explain what critical thinking means.","input":"","output":"Critical thinking is the ability to analyze information logically and make reasoned decisions."},
{"instruction":"Why is creativity important?","input":"","output":"Creativity helps generate new ideas and innovative solutions to challenges."},
{"instruction":"Explain what collaboration means.","input":"","output":"Collaboration is working together with others to achieve shared goals."},
{"instruction":"What is learning?","input":"","output":"Learning is the process of acquiring knowledge, skills, or understanding."},
{"instruction":"Explain what research means.","input":"","output":"Research is the systematic investigation of a topic to discover new knowledge."},

{"instruction":"What is space exploration?","input":"","output":"Space exploration is the investigation of outer space using spacecraft and technology."},
{"instruction":"Explain what a satellite is.","input":"","output":"A satellite is an object that orbits a planet and is often used for communication and observation."},
{"instruction":"What is the solar system?","input":"","output":"The solar system consists of the Sun and the objects that orbit it including planets and asteroids."},
{"instruction":"Explain what gravity is.","input":"","output":"Gravity is the force that attracts objects toward each other."},
{"instruction":"What is astronomy?","input":"","output":"Astronomy is the scientific study of celestial objects such as stars, planets, and galaxies."},

{"instruction":"Explain what data science is.","input":"","output":"Data science is a field that uses statistics, programming, and analysis to extract insights from data."},
{"instruction":"What is data visualization?","input":"","output":"Data visualization is the graphical representation of data using charts and graphs."},
{"instruction":"Explain what big data means.","input":"","output":"Big data refers to extremely large datasets that require advanced tools to analyze."},
{"instruction":"What is predictive analytics?","input":"","output":"Predictive analytics uses data and statistical techniques to forecast future outcomes."},
{"instruction":"Explain what statistics is.","input":"","output":"Statistics is the science of collecting, analyzing, and interpreting data."},

{"instruction":"Explain the importance of education.","input":"","output":"Education helps individuals gain knowledge and skills necessary for personal and professional growth."},
{"instruction":"What is digital literacy?","input":"","output":"Digital literacy is the ability to effectively use digital tools and technologies."},
{"instruction":"Explain lifelong learning.","input":"","output":"Lifelong learning is the continuous pursuit of knowledge throughout a person's life."},
{"instruction":"Why are libraries important?","input":"","output":"Libraries provide access to knowledge, research materials, and educational resources."},
{"instruction":"Explain what knowledge sharing means.","input":"","output":"Knowledge sharing is the process of exchanging information and expertise with others."}
]

dataset = Dataset.from_list(data)

**Configure 4-bit Quantization (QLoRA)**

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True
)

**Load Mistral Model**

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

**Configure LoRA**

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

**Training Configuration**

In [ ]:
training_args = SFTConfig(
    output_dir="mistral-qlora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=False,
    bf16=False
)

In [ ]:
def format_dataset(example):
    return {
        "text": f"### Instruction: {example['instruction']}\n### Response: {example['output']}"
    }

dataset = dataset.map(format_dataset)

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

**Create Trainer**

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/60 [00:00<?, ? examples/s]

**Train Model**

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,2.118966
10,1.153444
15,1.031267


TrainOutput(global_step=15, training_loss=1.434559154510498, metrics={'train_runtime': 77.7561, 'train_samples_per_second': 0.772, 'train_steps_per_second': 0.193, 'total_flos': 85409710080000.0, 'train_loss': 1.434559154510498})

**Save Model**

In [ ]:
trainer.model.save_pretrained("mistral-qlora-finetuned")
tokenizer.save_pretrained("mistral-qlora-finetuned")

('mistral-qlora-finetuned/tokenizer_config.json',
 'mistral-qlora-finetuned/tokenizer.json')

**Testing the Fine-Tuned Model**

In [ ]:
prompt = "### Instruction: What is phishing.\n### Response:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id
)

#print(tokenizer.decode(outputs[0], skip_special_tokens=True))

NameError: name 'tokenizer' is not defined

In [ ]:
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

response = decoded.split("### Response:")[-1]

print(response.strip())

Phi- #1. using the same as by one of ## How do you are a large, import React from  A A few people with By definition \text{x = Ded and The result, User: Culting for QUI_ In fact that B28). I don's to Hydrogenic species is no effect on It was a lot of #40)
 política de la in - - -  The United States will not be asked if it’ # An old;  #1950,  We have been the number of T
